# Model & Inquiry: The World Your Question Lives In

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/honr_464_logo.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb04_model_inquiry_student.ipynb)

---

## 🧭 Approach & Claim Boundary

**Approach emphasis:** causal reasoning + statistical inference (building the
**M**odel and defining the **I**nquiry — the first two letters of MIDA). Before
you can answer a causal question, you have to say what world it lives in and
exactly which number you are after. This notebook builds both.

| | |
|---|---|
| **A claim this topic PERMITS** | "Under my stated model of the world, the quantity I want exists and is defined — and here is that quantity, named before I saw any outcome data." |
| **A claim this topic does NOT permit** | "The effect of X" with no world in which X varies (a model you never stated) — and choosing the quantity *after* browsing results, then presenting it as the question you always had. |

**Where this sits in the course:** meetings 9–10 of 44. It develops milestone
M03 (your Model & Inquiry Declaration) and builds on the Literature & Claim Map
(M02, nb03): your gap named a question; now you specify the world that question
assumes and the exact quantity that would answer it.

*Provenance: RDSS ch. 6 'Specifying the model' + ch. 7 'Defining the inquiry' + declaration_7.1 + utilities/make_dag_df.R | ch. 6–7 | potential outcomes, the estimand, and a DAG helper | translated (R→Python) + newly-constructed-from-source-concept*

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain what a **model** is in the MIDA sense — a claim about how the world
   *could* be — and why every causal question hides one.
2. Read and write **potential outcomes** (Y0, Y1) and compute an individual
   treatment effect inside a world you can see completely.
3. State the **fundamental problem of causal inference** in your own words after
   executing it: we never observe both potential outcomes for the same unit.
4. Draw a three-node **DAG** from an edge list and say what each arrow asserts
   about the real world.
5. Define an **inquiry (estimand)** — the exact number a genie would hand you —
   and tell a descriptive inquiry from a causal one.
6. Declare the model and inquiry for your **own** project (milestone M03).

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)

DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

## 1. Why This Matters

> *"You keep telling me the mentoring program 'works.' Works compared to what?
> Compared to the same undergraduates in a world where they never got a mentor? Then
> show me that world — because that is the only comparison your claim is
> actually about."*
> — a **policy stakeholder** deciding whether to fund the program next year

Every causal claim is secretly a claim about two worlds: the world as it is, and
a world that did not happen. "The mentor raised her sense of belonging" means
*her belonging is higher than it would have been in the world where she had no
mentor* — a world you will never get to see. That invisible comparison world is
not a technicality. It is the entire meaning of the word "effect," and research
that cannot say clearly which two worlds it is comparing is not yet asking a
causal question at all.

MIDA — the framework this course runs on — starts here, with two letters:
**M**odel (what the worlds could look like) and **I**nquiry (the exact quantity
you want from them). This notebook builds a small world you can see *completely*,
including the parts reality always hides, so that when the hidden part disappears
you will feel exactly what makes causal inference hard — and exactly what you are
declaring when you write down your own model and inquiry for M03.

The master map from nb00 returns — the one you were promised you would OWN by
December. Today you start owning its first two letters, M and I:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_5_1.png" width="75%"/></center>

*The MIDA map, the same diagram you met in nb00: a research design has a Model, an
Inquiry, a Data strategy, and an Answer strategy (M·I·D·A), and this notebook
builds the first two. (Figure from the replication materials of Blair, Coppock &
Humphreys (2023),* Research Design in the Social Sciences *— MIT-licensed archive;
the book is free at book.declaredesign.org.)*

> **A question that often comes up here:** *"If the comparison world never happens,
> how can research ever measure an effect?"* That tension is the whole game, and
> it has an honest answer — but the answer starts with admitting the problem, not
> hiding it. Today we make the problem vivid. The rest of the causal units in this
> course are all different strategies for reasoning about the world you cannot
> see, each with its own price. You cannot appreciate the strategies until you
> have felt the problem.

---

## 2. Potential Outcomes: The Two Worlds Every Causal Question Hides

**Guiding question:** *when you say a treatment "had an effect," which two worlds
are you secretly comparing — and could you ever stand in both?*

Here is the vocabulary that makes the two-worlds idea precise. For each person,
imagine two numbers:

- **Y0** — the outcome they *would* have **without** the treatment.
- **Y1** — the outcome they *would* have **with** it.

These are called **potential outcomes**: both are real possibilities, but only
one of them will actually occur for any given person, depending on whether they
get the treatment. The person's own **individual treatment effect** is simply
**Y1 − Y0** — how much the treatment moved *their* outcome.

We will work in a concrete world all course long: the **mentoring-program
world**. Each person is a first-year undergraduate; the outcome is a
**sense-of-belonging score from 0 to 100**; the treatment is being paired with a
**peer mentor**. So Y0 is a person's belonging without a mentor, Y1 is their
belonging with one, and Y1 − Y0 is what the mentor did for them.

The trick we get to play — only because this is a simulation — is to build a
world where we can see **both** Y0 and Y1 for **every** person at once. Reality
never lets you do that. That is precisely why we start in simulation: to look at
the thing reality hides, name it, and then watch it vanish.

> **A question that often comes up here:** *"Is the mentor's effect the same for
> everyone?"* In reality, almost never — a mentor might lift one person's
> belonging a lot and barely move another's, so real individual effects vary,
> which is why we end up caring about the *average* effect across a group. The
> teaching world you are about to build deliberately sets that variation aside
> and gives everyone the same effect, for a reason that will become clear the
> moment you have both potential-outcome columns in front of you.

---

### 🔮 Pause & Predict

We are about to simulate 100 people and — because it is a simulation — print
**both** potential outcomes, Y0 and Y1, side by side for each one.

**Before running the next cell**, commit a prediction: in the *real* mentoring
program (not the simulation), for how many of the 100 people could you ever
directly observe **both** their with-mentor and without-mentor belonging score?
Write a number and one sentence on why.

### YOUR ANSWER HERE:

**Number of people for whom I could observe BOTH potential outcomes in reality:**

**Why:**

---

### 🛠️ Run the Study: build a world you can see completely

Run the cell below. It defines the mentoring-program world and simulates 100
people, showing you both potential outcomes and each person's individual
treatment effect — the god's-eye view that only a simulation allows.

> 💡 **Gemini Prompt:** "Here is a Python function from my research-methods
> course that builds a small simulated 'world' of 100 people, each with two
> potential outcomes — Y0 (belonging without a mentor) and Y1 (belonging with
> one, built as Y0 plus a fixed effect): [paste the next cell]. Walk me through
> what each line does, and explain why being able to see BOTH Y0 and Y1 for the
> same person is something only a simulation can offer."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check Gemini's line-by-line story against the printed table — do a Y0
>   column, a Y1 column, and an individual_effect column all actually appear?
> - [ ] Confirm its account of the built-in effect: does every person's
>   individual_effect really come out to the same +2.0 the code dialed in?
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# The mentoring-program world. This exact function is reused later in the
# course (nb09-nb11), so read it carefully now.
def make_world(n=100, effect=2.0, noise=2.0, rng=rng):
    """The mentoring-program world: each person's belonging score (0-100)
    without a mentor (Y0) and with one (Y1 = Y0 + effect)."""
    Y0 = rng.normal(50, 10, n) + rng.normal(0, noise, n)
    return pd.DataFrame({"Y0": Y0.round(1), "Y1": (Y0 + effect).round(1)})

world = make_world()
world["individual_effect"] = (world["Y1"] - world["Y0"]).round(1)

print("✓ Simulated a 100-person mentoring-program world (both worlds visible).")
print("\nFirst 8 people — the god's-eye view:")
print(world.head(8).to_string())
print(f"\nIn THIS world, every person's individual effect is the same:"
      f" {world['individual_effect'].iloc[0]} points — by deliberate construction.")

### 🔍 Reading the Evidence

In the cell below, write two things about the table you just saw. First: this
world moves *every* person by exactly +2.0 points — a deliberately simple
construction. Name two concrete reasons a real mentoring program would NOT move
everyone equally (think about who the people are and how a mentor actually
helps). Second — the important one — write the single sentence that captures
*why this table could never exist for a real mentoring program*. Be specific
about which numbers you could and could not actually collect.

> **A question that often comes up here:** *"If real effects vary person to
> person, why build a world where they don't?"* Because this world's job is to
> be an **answer key**. In the coming units you will hand this world to
> estimators and simulations and ask how close they land to the truth — and for
> that, the truth needs to be a single crisp number. In a richer world the
> realized average effect for these particular 100 people would drift away from
> the +2.0 dial (each sample of people would carry its own average), and you
> will meet exactly that idea when nb10 takes on generalization. Simple first,
> rich later — on purpose.

### YOUR ANSWER HERE:

**Two reasons a real program's effects would vary from person to person:**

**Why this exact table could never exist for a real program:**

---

## 3. The Fundamental Problem of Causal Inference

**Guiding question:** *once each person can show you only one of their two
potential outcomes, what exactly is it that becomes impossible to compute?*

Now we do the thing reality does. Each person either gets a mentor or does not —
so for each person, **one** of their two potential outcomes becomes the number
you actually observe, and the other silently vanishes into the world that did not
happen. Run the next cell to flip a coin for each person and keep only the
outcome that "really occurred."

The result is the table every real study is stuck with: a column of observed
outcomes, and a ghost column of question marks where the counterfactual used to
be. The `individual_effect` you computed a moment ago becomes **uncomputable** —
there is no person for whom you still hold both numbers.

> 💡 **Gemini Prompt:** "This cell takes a simulated world where I can see both
> potential outcomes and then does what reality does — it assigns each person to
> mentor or not by a coin flip, keeps only the outcome that matches the
> assignment, and hides the other: [paste the next cell]. Explain what np.where
> is doing on each line, and why the individual_effect column becomes a column of
> question marks afterward."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check that after this cell runs, each row shows a value in EITHER
>   Y0_seen or Y1_seen — never both — matching that person's Z.
> - [ ] Confirm the mentored and not-mentored counts printed below add up to the
>   full 100 people; the coin split need not land at exactly 50/50.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# Reality's move: assign each person to mentor (Z=1) or not (Z=0) by a coin,
# then observe ONLY the potential outcome that matches their assignment.
world["Z"] = rng.integers(0, 2, len(world))                      # the coin
world["observed"] = np.where(world["Z"] == 1, world["Y1"], world["Y0"])

# What a real study actually gets to see: assignment, the observed outcome, and
# a ghost column where the counterfactual is now unknowable.
real_study = world[["Z", "observed"]].copy()
real_study["Y0_seen"] = np.where(world["Z"] == 0, world["Y0"], np.nan)
real_study["Y1_seen"] = np.where(world["Z"] == 1, world["Y1"], np.nan)
real_study["individual_effect"] = "?"   # can never be filled in

print("What a real study is stuck with (first 8 people):")
print(real_study.head(8).to_string())
print(f"\nMentored: {int(world['Z'].sum())} people   "
      f"Not mentored: {int((world['Z'] == 0).sum())} people")
print("\nThe individual_effect column is now unknowable for every single person.")

That missing column is not a data-collection failure you could fix with a bigger
budget or a better survey. It is a logical wall: a person cannot both have and
not have a mentor this semester. This is the **fundamental problem of causal
inference** — for any unit, at most one potential outcome is ever observable.
Every causal method in the rest of this course is, at heart, a principled way of
reasoning about that missing column *without* pretending you can see it. Naming
the true average effect, and later trying to recover it, is the story of the next
several units. First, though, we need a way to draw the beliefs that justify any
such reasoning.

---

### 📝 Practice

For each question below, name the **potential-outcomes pair** it implicitly
compares — the "with" world and the "without" world for the unit. Write each as
"Y1 = … ; Y0 = …".

- **A.** "Does a text-message reminder increase the chance a voter turns out?"
- **B.** "Does switching a course to pass/fail change how many hours undergraduates
  study for it?"
- **C.** "Did the new campus shuttle reduce a commuter's average travel time?"


### YOUR ANSWER HERE:

**A — Y1 / Y0:**

**B — Y1 / Y0:**

**C — Y1 / Y0:**

---

## 4. DAGs: Drawing the Arrows You Believe

**Guiding question:** *how do you write a belief about what causes what down on
paper — clearly enough that the world could prove you wrong?*

A **DAG** (directed acyclic graph) is the simplest honest way to write down a
model — your belief about what could affect what. Each **node** is a variable;
each **arrow** is a claim that one variable *could* directly influence another.
"Acyclic" just means no variable ends up causing itself through a loop. The
smallest honest picture has just two nodes and one arrow — a treatment Z pointing
at an outcome Y — plus an acknowledgment that units differ in ways you have not
named:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_6_1.png" width="55%"/></center>

*A minimal DAG: the treatment Z points at the outcome Y, with U standing for the
unknown ways people differ. (Figure from the replication materials of Blair,
Coppock & Humphreys (2023),* Research Design in the Social Sciences *— MIT-licensed
archive; the book is free at book.declaredesign.org.)*

The reason a picture earns its place: the moment you draw the arrows, you commit
to a story the world could confirm or contradict — and you expose the variables
that quietly threaten your causal claim. In the mentoring world, suppose undergraduates
who were **already more involved on campus** are both more likely to seek out a
mentor *and* more likely to feel they belong anyway. Then "prior involvement" is
a common cause of both the treatment and the outcome — a **confounder** — and any
naive comparison of mentored versus unmentored people is partly measuring
involvement, not the mentor. A DAG makes that danger visible as a shape you can
point at. Run the next cell to draw exactly this three-node model.

> 💡 **Gemini Prompt:** "Here is Python that draws a three-node DAG from an edge
> list using networkx — the nodes are prior involvement, peer mentor, and
> belonging: [paste the next cell]. Explain what a directed edge asserts about
> the world, and tell me in plain terms why 'prior involvement' pointing at BOTH
> 'peer mentor' and 'belonging' is what makes it a confounder."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the drawn graph against the edges list: are there exactly three
>   arrows, and do two of them start at 'prior involvement'?
> - [ ] Confirm Gemini's confounder explanation matches the picture — a
>   confounder has an arrow into the treatment AND an arrow into the outcome.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# A model of the mentoring world, drawn from an edge list. Each arrow is a
# belief: "this variable could directly affect that one."
import networkx as nx

edges = [
    ("prior involvement", "peer mentor"),   # already-involved people seek mentors
    ("prior involvement", "belonging"),     # ...and tend to feel they belong anyway
    ("peer mentor",       "belonging"),      # the causal arrow we actually care about
]
dag = nx.DiGraph(edges)

pos = nx.spring_layout(dag, seed=464)   # seed keeps the layout reproducible
fig, ax = plt.subplots(figsize=(8, 5))
nx.draw_networkx(dag, pos=pos, ax=ax, node_size=3000, node_color="#dbe4ff",
                 edgecolors="#3b5bdb", linewidths=1.5, font_size=10,
                 arrowsize=22, width=1.5)
ax.set_title("A three-node model of the mentoring world "
             "(prior involvement confounds mentor → belonging)")
ax.set_axis_off()
fig.tight_layout()
plt.show()

print("Nodes:", list(dag.nodes()))
print("Arrows (each is a claim about the real world):", list(dag.edges()))

A confounder is only the first of a small zoo of arrow patterns, and each one
carries a different threat (or opportunity) for a causal claim. You do not need
to master all of them today, but you should recognize the shapes when you meet
them again in October:

<center><img src="https://raw.githubusercontent.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/main/notebooks/figures/rdss_fig_6_2.png" width="80%"/></center>

*The vocabulary of arrow patterns: a confounder, a moderator, a mediator, a
collider, and an instrument — five roles a third variable can play between a
treatment and an outcome. (Figure from the replication materials of Blair,
Coppock & Humphreys (2023),* Research Design in the Social Sciences *— MIT-licensed
archive; the book is free at book.declaredesign.org.)*

The one to hunt for now is the **confounder** on the left — the common cause your
own DAG needs to expose.

---

### ⚖️ Make a Design Choice

Your DAG is an argument, and every arrow is contestable. In the cell below, do
two things. First, name **one more** variable that could plausibly be a confounder
in the mentoring world (a common cause of both getting a mentor and feeling
belonging) — and decide whether to add it to the model, defending the call in a
sentence. Second, pick the single arrow in your DAG you are **least** sure about
and state what real-world observation would make you erase it.

> 💡 **Gemini Prompt:** "I am modeling whether [your
> treatment] affects [your outcome] for [your units]. List variables that could
> be common causes of both — potential confounders I should consider putting in
> my DAG — and for each, state the mechanism by which it would influence both."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] For each suggested confounder, decide *yourself* whether the mechanism is
>   real for your context — the AI is brainstorming candidates, not settling facts.
> - [ ] Confirm each is genuinely a *common cause* (points to both treatment and
>   outcome), not just something correlated — draw the arrows and check.
> - [ ] Log this use in your AI ledger: tool, task, verification.

### YOUR ANSWER HERE:

**One more candidate confounder + whether I add it (and why):**

**The arrow I trust least + what observation would make me erase it:**

---

### 🎯 Project Transfer — draw your project's DAG

Adapt the DAG cell above to your **own** project. Replace the three node labels
with your treatment (or main explanatory variable), your outcome, and one lurking
third variable that could affect both. Copy the code into the cell below, edit the
`edges` list, and run it — then read the picture back to yourself: does every
arrow assert something you actually believe about the world?

### YOUR SOLUTION HERE
# Paste the DAG code from above and edit the three node labels + edges list
# to match your own project's world. Then run it.


### 🎟️ Claim Ticket

**Claim Ticket #9** — in the cell below, name **one arrow** in your DAG and the
real-world mechanism it asserts (the actual story by which the first variable
could move the second). One sentence.

### YOUR ANSWER HERE:

**My arrow + the real-world mechanism it asserts:**

---

---

*(Meeting M10 picks up here.)*

## 5. The Inquiry: The Genie Test

> *"Before you collect a single data point, tell me the exact number you are
> trying to learn. If a genie appeared and offered you the truth, what would you
> ask it for? If you cannot say, you are not ready to gather data — you are ready
> to go fishing."*
> — a **thesis advisor**, saving you from a semester of aimless analysis

The **inquiry** (also called the **estimand**) is that number: the precise
quantity a genie with perfect knowledge would hand you. It is the **I** in MIDA,
and defining it is a separate act from building the model. In the mentoring
world, the most common inquiry is the **Average Treatment Effect (ATE)** — the
average of every person's individual effect across the whole world:

**ATE = average of (Y1 − Y0) over all units.**

Because we built this world in a simulation, we are in the genie's seat: we hold
every Y0 and every Y1, so we can compute the ATE exactly. Real studies never can
— which is exactly why the inquiry has to be *declared*, in words and as a
formula, before the data arrive. Declaring it first is what stops you from
quietly redefining "the effect" later to match whatever your results happened to
show.

> **A question that often comes up here:** *"Why write a formula when I can just
> say 'the effect'?"* Because "the effect" is ambiguous and a formula is not. The
> average effect over everyone, the effect just for people who chose the mentor,
> the share of people helped at all — these are different numbers with different
> formulas, and they can point in different directions. The formula forces you to
> pick one *before* the data can tempt you toward whichever looks best.

---

### 🔮 Pause & Predict

Look back at the `make_world` function: it builds `Y1 = Y0 + effect`, with
`effect=2.0` — a nudge of +2 points wired into the machine.

**Before running the next cell**, predict: when we compute the *actual* ATE — the
average of the 100 individual effects in this specific world — will it come out
**exactly 2.0**, **a little off from 2.0**, or **far from 2.0**? One sentence on
why — and be ready to defend it from the function's own code. (This is a
read-the-machine question: the answer is sitting in the line that builds Y1.)

### YOUR ANSWER HERE:

**The computed ATE will be (exactly 2.0 / a little off / far off):**

**Why:**

---

> 💡 **Gemini Prompt:** "This cell computes the 'true' average treatment effect
> in a simulated world by averaging Y1 minus Y0 over all 100 people, then argues
> the number becomes hidden once you keep only one outcome per person: [paste the
> next cell]. Explain why the average comes out to exactly the effect the world
> was built with, and what the phrase 'the ATE is the answer key' means here."
>
> **After running, verify (the responsible-AI habit):**
> - [ ] Check the printed TRUE ATE against the +2.0 the world dialed in — the
>   assert line guarantees the match, so confirm the value really prints as 2.0.
> - [ ] Make sure you can say WHY it is exact (every person gets the same effect),
>   not just that it is — Gemini should point you to the Y1 = Y0 + effect line.
> - [ ] Log this use in your AI ledger: tool, task, verification.

In [ ]:
# We are the genie: compute the exact inquiry value for THIS world.
TRUE_ATE = (world["Y1"] - world["Y0"]).mean()

print(f"The dialed-in nudge was +2.0 points per person.")
print(f"The TRUE ATE realized in these 100 people is: {TRUE_ATE:.3f}")

# Self-check: constant-effect construction means the ATE equals the dial exactly.
assert abs(TRUE_ATE - 2.0) < 0.01, "constant-effect world must realize ATE = 2.0"
print("\n✓ This number is the ANSWER KEY.")
print("We now 'hide' it: from the observed (one-outcome-per-person) data alone,")
print("no one can recompute it — later units will try to RECOVER it with estimators.")

# Prove it is hidden: the observed data cannot rebuild the individual effects.
can_compute_from_observed = world[["Z", "observed"]].assign(effect="unknowable")
print("\nFrom observed data alone, every individual effect stays:",
      can_compute_from_observed["effect"].unique()[0])

### 🔍 Reading the Evidence

In the cell below, two things. First: the ATE came out exactly 2.0 — say, in
your own words, what feature of the machine guaranteed that (and what would
have to change for it NOT to be guaranteed — look back at the heterogeneity
Q&A). Second, the question that matters for your own project: the ATE is a
fact about the *specific* units in a world. In a richer world where effects
vary person to person, what does that imply about an effect measured in one
sample of 100 people versus the effect in the whole population those people
came from? (You are previewing nb10; a hunch is enough.)

### YOUR ANSWER HERE:

**What guaranteed exactly 2.0 — and what would break the guarantee:**

**What this implies about a one-sample effect vs the whole-population effect:**

---

## 6. Descriptive vs Causal Inquiries

**Guiding question:** *how can you tell, from the formula alone, whether your
question needs a world that never happened?*

Not every inquiry is a treatment effect. The genie test works for any quantity;
the question is just *which* quantity you ask for. Two broad families:

- A **descriptive inquiry** asks about the world **as it is** — no second world
  required. "What is the average belonging score?" is the mean of Y. "What share
  of undergraduates score above 60?" is a proportion. These need only the outcomes you
  can, in principle, observe.
- A **causal inquiry** asks about the **difference between two worlds** — it needs
  potential outcomes. The ATE is the flagship example: average of Y1 − Y0.

The tell is whether the quantity *requires a world that did not happen*. A
description never does; a causal inquiry always does. That single distinction
decides which of the four approaches your question belongs to, how hard the
answer will be to earn, and what could go wrong on the way. Writing your inquiry
as a formula makes the family obvious: if a counterfactual (Y0 or Y1 for a unit
that did not get that condition) appears in the formula, you are making a causal
claim and owe the world that justifies it.

---

### 📝 Practice

For each inquiry below, (i) write it as a **formula** over outcomes or potential
outcomes, and (ii) label it **descriptive** or **causal**.

- **A.** "The average sense-of-belonging score among all first-year undergraduates."
- **B.** "The average effect of the peer-mentor program on belonging."
- **C.** "The share of undergraduates who feel they belong (score above 60)."
- **D.** "How much higher a person's belonging is with a mentor than it would
  have been without one, averaged over people."


### YOUR ANSWER HERE:

**A — formula / type:**

**B — formula / type:**

**C — formula / type:**

**D — formula / type:**

---

### 🎯 Project Transfer — declare your model and inquiry (M03)

This is the heart of milestone M03. In the cell below, assemble the declaration
you will present: (1) your **model** in one or two sentences (the world your
question assumes, plus the one confounder your DAG flagged), and (2) your
**inquiry** stated the genie way — one plain-English sentence *and* a formula —
labeled descriptive or causal. Declaring it now, before your data strategy, is
the discipline that protects the rest of your project from inquiry-shopping.

### YOUR ANSWER HERE:

**My model (the world + the key confounder from my DAG):**

**My inquiry, in plain English (the genie sentence):**

**My inquiry, as a formula:**

**Descriptive or causal (and how the formula tells me):**

---

### 🎟️ Claim Ticket

**Claim Ticket #10** — in the cell below, state your inquiry in **one sentence a
non-researcher could repeat back to you** without using a formula or jargon. If a
friend could not restate it, it is not ready.

### YOUR ANSWER HERE:

**My inquiry, one plain sentence:**

---

## 7. Wrap-Up

Across two meetings you assembled the first half of MIDA. You learned that every
causal claim compares two worlds, wrote those worlds down as potential outcomes,
and then executed the fundamental problem of causal inference by watching one
potential outcome per person vanish into the counterfactual you can never see.
You drew a model as a DAG, where every arrow is a contestable claim and a common
cause is a visible threat. And you defined an inquiry with the genie test — the
exact number, in words and formula, declared *before* the data can tempt you —
and told a description apart from a causal effect by asking whether the formula
needs a world that did not happen.

> **"A causal question you cannot draw as two worlds and write as one formula is
> not yet a question — it is a wish. The model says which worlds; the inquiry says
> which number; only then are you allowed to go looking."**

On Friday, nb05 takes the quantity you just declared and asks the next hard
question: how do you turn the abstract construct in your inquiry into a number you
can actually collect — and how wrong can that number be before your answer
changes? Bring your M03 declaration; you present it Wednesday, and it is the
foundation everything after it stands on.

---

## 8. Sources & Provenance

**Provenance of this notebook:**
- *RDSS ch. 6 'Specifying the model' | the M of MIDA | possible worlds + potential outcomes Y0/Y1 | translated (R→Python, honors non-quant framing)*
- *RDSS ch. 7 'Defining the inquiry' + declaration_7.1 | the I of MIDA | the estimand / ATE and the descriptive-vs-causal split | translated (R→Python)*
- *utilities/make_dag_df.R | DAG helper | edge-list DAG drawn with networkx spring_layout (seed 464) | translated (R→Python)*
- *fresh | the mentoring-program world simulation (seed 464) + the genie-test framing | — | newly-constructed-from-source-concept*

**Readings:**
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the
  Social Sciences*, ch. 6–7. Free online:
  [book.declaredesign.org](https://book.declaredesign.org/).

---

<center>

Thank you!

</center>